# Calendar Dimension Build

Creates the standard calendar table `adb_caresync.silver.calendar` covering every date from 1940-01-01 to 2023-12-31, with date key, year, quarter, month, week, day and flag attributes.

In [0]:
CREATE OR REPLACE TABLE adb_caresync.silver.calendar
COMMENT 'Standard calendar dimension: one row per day from 1940-01-01 to 2023-12-31'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2023-12-31', INTERVAL 1 DAY)) AS date
)
SELECT
  -- Keys
  CAST(date_format(date, 'yyyyMMdd') AS INT)              AS date_key,
  date                                                    AS date,

  -- Year
  year(date)                                              AS year,
  CASE WHEN (year(date) % 4 = 0 AND year(date) % 100 <> 0) OR year(date) % 400 = 0
       THEN true ELSE false END                           AS is_leap_year,
  trunc(date, 'YEAR')                                     AS first_day_of_year,
  make_date(year(date), 12, 31)                           AS last_day_of_year,

  -- Half year
  CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END            AS half_year,
  CASE WHEN month(date) <= 6 THEN 'H1' ELSE 'H2' END      AS half_year_name,

  -- Quarter
  quarter(date)                                           AS quarter,
  concat('Q', quarter(date))                              AS quarter_name,
  concat(year(date), '-Q', quarter(date))                 AS year_quarter,
  trunc(date, 'QUARTER')                                  AS first_day_of_quarter,
  date_sub(add_months(trunc(date, 'QUARTER'), 3), 1)      AS last_day_of_quarter,
  datediff(date, trunc(date, 'QUARTER')) + 1              AS day_of_quarter,

  -- Month
  month(date)                                             AS month,
  date_format(date, 'MMMM')                               AS month_name,
  date_format(date, 'MMM')                                AS month_short_name,
  date_format(date, 'yyyy-MM')                            AS year_month,
  CAST(date_format(date, 'yyyyMM') AS INT)                AS year_month_key,
  trunc(date, 'MM')                                       AS first_day_of_month,
  last_day(date)                                          AS last_day_of_month,
  day(last_day(date))                                     AS days_in_month,

  -- Week (ISO 8601: weeks start Monday)
  weekofyear(date)                                        AS week_of_year,
  extract(YEAROFWEEK FROM date)                           AS iso_year,
  CAST(ceil(day(date) / 7.0) AS INT)                      AS week_of_month,
  CAST(date_trunc('WEEK', date) AS DATE)                  AS week_start_date,
  date_add(CAST(date_trunc('WEEK', date) AS DATE), 6)     AS week_end_date,

  -- Day
  day(date)                                               AS day_of_month,
  dayofyear(date)                                         AS day_of_year,
  dayofweek(date)                                         AS day_of_week,      -- 1 = Sunday ... 7 = Saturday
  extract(DAYOFWEEK_ISO FROM date)                        AS iso_day_of_week,  -- 1 = Monday ... 7 = Sunday
  date_format(date, 'EEEE')                               AS day_name,
  date_format(date, 'EEE')                                AS day_short_name,

  -- Flags
  dayofweek(date) IN (1, 7)                               AS is_weekend,
  dayofweek(date) NOT IN (1, 7)                           AS is_weekday,
  date = trunc(date, 'MM')                                AS is_month_start,
  date = last_day(date)                                   AS is_month_end,
  date = trunc(date, 'QUARTER')                           AS is_quarter_start,
  date = date_sub(add_months(trunc(date, 'QUARTER'), 3), 1) AS is_quarter_end,
  date = trunc(date, 'YEAR')                              AS is_year_start,
  date = make_date(year(date), 12, 31)                    AS is_year_end
FROM dates
ORDER BY date;

In [0]:
-- Expect 30,681 rows (84 years x 365 + 21 leap days), unique date_key, full range
SELECT
  count(*)                 AS row_count,
  count(DISTINCT date_key) AS distinct_date_keys,
  min(date)                AS min_date,
  max(date)                AS max_date,
  sum(CAST(is_leap_year AND month = 2 AND day_of_month = 29 AS INT)) AS leap_days
FROM adb_caresync.silver.calendar;

In [0]:
SELECT * FROM adb_caresync.silver.calendar ORDER BY date LIMIT 10;